# CSE 5509 Final Project: Ego-Centered 360° Minimap from Single-Camera Views

This notebook demonstrates an approximate ego-centered minimap/radar pipeline that combines four computer vision tasks:
1) pretrained semantic segmentation,
2) pretrained monocular depth estimation,
3) pretrained object detection,
4) geometric projection into a shared BEV/minimap coordinate frame.

**Capture convention used in this project**: at each location, the camera stays at one fixed point; direction index increases by turning **right** by about 45° each step (`direction 0` ... `direction 7`).


In [ ]:
from pathlib import Path
import json
import sys
import importlib

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
except Exception as e:
    print(f"Colab Drive mount skipped: {e}")

PROJECT_DIR = Path("/content/drive/MyDrive/CSE 5509 Final Project")  # edit only this if needed
PROJECT_DIR = PROJECT_DIR.expanduser().resolve()

if not (PROJECT_DIR / "bev_pipeline.py").exists():
    raise FileNotFoundError(f"bev_pipeline.py not found in PROJECT_DIR: {PROJECT_DIR}")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import bev_pipeline
importlib.reload(bev_pipeline)

from bev_pipeline import PipelineConfig, resolve_project_paths

paths = resolve_project_paths(PROJECT_DIR)
CONFIG = PipelineConfig(
    repo_root=paths["repo_root"],
    data_dir=paths["data_dir"],
    output_dir=paths["output_dir"],
    run_small_demo=False,
    demo_locations=1,
    demo_images_per_location=None,
    horizontal_fov_deg=76.0,  # Xiaomi 13 1x main camera starting estimate
    min_depth_m=2.0,
    max_depth_m=40.0,
    minimap_size_px=1400,
    minimap_max_distance_m=40.0,
    minimap_scale_px_per_m=None,
    minimap_draw_dense_points=True,
    minimap_draw_object_labels=True,
    minimap_min_confidence=0.75,
    minimap_label_top_k_per_location=60,
    minimap_marker_alpha=0.85,
    minimap_jitter_overlapping_markers=True,
    minimap_merge_nearby_same_class=True,
    minimap_merge_radius_m=1.5,
    direction_zero_heading_deg=0.0,
    direction_step_deg=45.0,
    direction_turn="right",
    clean_output_dir=False,
)
print(f"Repo root : {paths['repo_root']}")
print(f"Data dir  : {paths['data_dir']}")
print(f"Output dir: {paths['output_dir']}")
print(f"bev_pipeline.py found: {(paths['repo_root'] / 'bev_pipeline.py').exists()}")



## 3. Dataset discovery


In [ ]:
from bev_pipeline import discover_dataset
summary = discover_dataset(CONFIG.data_dir)
print(json.dumps({
    'location_count': summary['location_count'],
    'total_images': summary['total_images'],
    'images_per_location': summary['images_per_location'],
}, indent=2))


## 4. Load pretrained models


In [ ]:
from bev_pipeline import default_model_states, default_device

DEVICE = default_device()
model_states = default_model_states(device=DEVICE)
print('Device:', DEVICE)
print('Model availability:', {k: v.get('available', False) for k, v in model_states.items()})
for k, v in model_states.items():
    if not v.get('available', False):
        print(f"[{k}] fallback reason: {v.get('reason')}")

print("Note: Some Hugging Face warnings about legacy settings can appear and are non-fatal when models are available.")


## 5. Run pipeline


In [ ]:
from bev_pipeline import run_pipeline

run_report = run_pipeline(CONFIG, summary, model_states)
print('Saved run report:', CONFIG.output_dir / 'run_report.json')
print('Locations processed:', run_report['locations_processed'])
print('Images processed   :', run_report['images_processed'])
print('Total detections   :', run_report['total_detections'])


## 6. Display final minimap result


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

first_loc = run_report['location_results'][0]
minimap_path = Path(first_loc['minimap'])
plt.figure(figsize=(8,8))
plt.imshow(Image.open(minimap_path))
plt.axis('off')
plt.title(f"Primary output: {minimap_path.name}")
plt.show()


## 7. Display direction convention diagnostic


In [ ]:
debug_path = Path(run_report['location_results'][0]['direction_debug_plot'])
plt.figure(figsize=(8,8))
plt.imshow(Image.open(debug_path))
plt.axis('off')
plt.title('Direction convention check: dir0 up, dir1 up-right, dir2 right')
plt.show()


## 8. Display intermediate diagnostics


In [ ]:
import pandas as pd

first_image = run_report['location_results'][0]['images'][0]
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(Image.open(first_image['det_path']))
ax[0].set_title('Detection overlay')
ax[0].axis('off')
ax[1].imshow(Image.open(first_image['bev_path']))
ax[1].set_title('Per-image BEV diagnostic')
ax[1].axis('off')
plt.show()

table_path = Path(run_report['location_results'][0]['minimap_instance_table_csv'])
df = pd.read_csv(table_path)
print('Sample minimap instance table rows:')
display(df.head(12))


## 9. Concise run summary


In [ ]:
primary_paths = [r['minimap'] for r in run_report['location_results']]
print(json.dumps({
    'locations_processed': run_report['locations_processed'],
    'images_processed': run_report['images_processed'],
    'total_detections': run_report['total_detections'],
    'primary_minimap_outputs': primary_paths,
    'diagnostic_dir': run_report['output_dirs']['diagnostics'],
    'tables_dir': run_report['output_dirs']['tables'],
}, indent=2))


## 10. Assumptions, limitations, and AI usage

- This is an approximate single-camera ego-centered minimap visualization, not a calibrated autonomous-vehicle BEV system.
- Monocular depth has scale ambiguity; metric distances shown are approximate.
- Overlap across adjacent directions can produce repeated detections of the same real object.
- AI tooling was used for coding and documentation assistance; final outputs and claims were manually reviewed.

- Monocular depth is approximate and not metric ground truth.
- Open-vocabulary detection can hallucinate or miss objects.
- Deduplication is heuristic because there is no calibrated 3D reconstruction.
